# Part 1 Kaggle GPU backend — called by the local agent orchestrator

This notebook hosts the **GPU inference service only**: SDXL Inpainting + Canny ControlNet + LoRA, with optional Grounding DINO + SAM automatic masking. The LangGraph planner, worker orchestration, task scheduling, and evaluation run in the local application from the updated ZIP; they do not need to be duplicated in Kaggle.

**Compatibility contract:** the local application calls `GET /health`, `POST /v1/mask`, and `POST /v1/edit`. This notebook keeps those backend routes and the `X-Backend-Token` authentication header. Do not add the LangGraph orchestration layer to this notebook.

This update removes stale manual-debug cells and makes the secret check consistent with the startup code. The model-loading and inference implementation is otherwise preserved.


### Part 1 — Kaggle GPU Inference Backend (Automatic Masking + SDXL Inpainting)
This notebook deploys the existing SDXL Inpainting + Canny ControlNet + LoRA adapter as an authenticated HTTPS service for the local image-editing application. It does not train the adapter.
## End-to-end edit flow
1. Receive request: /v1/edit accepts the source image, edit instruction, operation, target description, and an optional user-provided mask.
2. Find target: if no mask is provided and auto_mask=true, Grounding DINO detects the requested existing object or placement region.
3. Create mask: SAM segments the selected detection; the generated white-on-black mask is used for the edit. /v1/mask is available to preview this mask independently.
4. Generate edit: SDXL Inpainting uses the source, mask, Canny ControlNet image, prompt, and loaded LoRA adapter.
5. Return result: the service responds with a PNG image. Original-resolution pixels outside the mask are copied back exactly, and response headers identify the mask mode, selected target, operation, and seed.
6. Fail clearly: invalid input, missing target, detection/segmentation failures, invalid masks, and GPU memory errors return meaningful HTTP errors. A failed mask/edit does not return a fake success image.
## Before running
1. In Kaggle, enable a GPU and internet access.
2. Attach the dataset architoff1/part1-sdxl-deployment-artifacts.
3. Add Kaggle Secrets NGROK_AUTHTOKEN and GPU_BACKEND_TOKEN. HF_TOKEN is optional for public model repositories.
4. Run the notebook from top to bottom and keep the Kaggle session running while the local app uses it.
5. Configure the local app with GPU_BACKEND_URL=<PUBLIC_URL>/v1/edit and the same GPU_BACKEND_TOKEN.
## Target semantics
- modify/replace: target names the object that already exists; prompt describes the desired result.
- add: target should name the existing placement region (for example, person's face when adding sunglasses); prompt describes the new object/result.
- remove: target names the object to remove; the backend prompts for a seamless continuation of the surrounding background.
- A provided mask always takes precedence over automatic masking. If target detection fails, the endpoint returns HTTP 422 rather than silently editing an unrelated region.
## Model-quality caveat
The supplied lora_step_0020.safetensors is a 20-step diagnostic adapter, not a production-validated model. The notebook implements the serving and automatic-mask workflow, but image quality depends on the checkpoint, target detection, mask quality, and prompt. Kaggle sessions and ngrok URLs are temporary; this is not persistent hosting.

## 1. Install serving dependencies

In [ ]:
%pip uninstall -y torchao
# Run before importing Diffusers/PEFT/FastAPI in this notebook.
%pip -q install "diffusers>=0.32.2,<0.42" "transformers>=4.45,<5.1" "accelerate>=1.0" "peft>=0.13" safetensors fastapi "uvicorn[standard]" python-multipart pyngrok pillow opencv-python-headless huggingface_hub requests

## 2. Locate and verify deployment artifacts

In [ ]:
from pathlib import Path
import hashlib, json, os, time, threading, secrets, io, gc
from getpass import getpass

import torch
import diffusers
import peft
import numpy as np
import cv2
from PIL import Image

ARTIFACT_ROOT = Path("/kaggle/input/datasets/architoff1/part1-sdxl-deployment-artifacts")
LORA_PATH = ARTIFACT_ROOT / "checkpoints/lora_step_0020.safetensors"
METRICS_PATH = ARTIFACT_ROOT / "evaluation/validation_metrics.csv"
REPORT_PATH = ARTIFACT_ROOT / "evaluation/validation_report.json"

if not LORA_PATH.is_file():
    raise FileNotFoundError(f"LoRA checkpoint not found: {LORA_PATH}. Attach the deployment dataset in Kaggle.")
expected_size = 46_607_400
if LORA_PATH.stat().st_size != expected_size:
    raise RuntimeError(f"Unexpected checkpoint size: {LORA_PATH.stat().st_size} bytes (expected {expected_size}).")

sha256 = hashlib.sha256()
with LORA_PATH.open("rb") as f:
    for chunk in iter(lambda: f.read(1024 * 1024), b""):
        sha256.update(chunk)
actual_sha256 = sha256.hexdigest()
expected_sha256 = "998a607ea55933fb72186a416ec4ddeb282d9211c8c3d3521ef4c804f4f6e1ca"
if actual_sha256 != expected_sha256:
    raise RuntimeError("Checkpoint SHA-256 mismatch; do not load an unexpected adapter.")

# The uploaded manifest/config placeholders were empty, so create corrected copies in writable storage.
OUTPUT_ROOT = Path("/kaggle/working/part1_sdxl_deployment")
METADATA_DIR = OUTPUT_ROOT / "metadata"
METADATA_DIR.mkdir(parents=True, exist_ok=True)
MODEL_CONFIG = {
    "base_model": "diffusers/stable-diffusion-xl-1.0-inpainting-0.1",
    "controlnet_model": "diffusers/controlnet-canny-sdxl-1.0",
    "adapter_type": "LoRA",
    "adapter_filename": LORA_PATH.name,
    "adapter_training_steps": 20,
    "adapter_status": "diagnostic_checkpoint_not_production_validated",
    "torch_dtype": "float16",
    "inference_resolution": 512,
}
ARTIFACT_MANIFEST = {
    "artifact_name": "part1-sdxl-deployment-artifacts",
    "checkpoint": {"filename": LORA_PATH.name, "size_bytes": LORA_PATH.stat().st_size, "sha256": actual_sha256},
    "validation_report": str(REPORT_PATH),
    "validation_metrics": str(METRICS_PATH),
    "note": "Validation pixel metrics do not establish semantic editing quality or photorealism."
}
(METADATA_DIR / "model_config.json").write_text(json.dumps(MODEL_CONFIG, indent=2) + "\n")
(METADATA_DIR / "artifact_manifest.json").write_text(json.dumps(ARTIFACT_MANIFEST, indent=2) + "\n")

print("PyTorch:", torch.__version__)
print("Diffusers:", diffusers.__version__)
print("PEFT:", peft.__version__)
print("Checkpoint:", LORA_PATH)
print("Checkpoint size:", LORA_PATH.stat().st_size, "bytes")
print("SHA-256 verified:", actual_sha256)
for p in (METRICS_PATH, REPORT_PATH):
    print(f"{'OK' if p.is_file() else 'OPTIONAL MISSING'}: {p}")
if REPORT_PATH.is_file() and REPORT_PATH.stat().st_size:
    try:
        print("Previous validation report:", json.dumps(json.loads(REPORT_PATH.read_text()), indent=2))
    except json.JSONDecodeError:
        print("Validation report is not valid JSON; it is not required for serving.")


## 3. GPU pre-flight and secrets

In [ ]:
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("GPU unavailable. Enable a GPU accelerator in Kaggle settings.")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    free, total = torch.cuda.mem_get_info(i)
    print(f"GPU {i}: {props.name} | total VRAM {total / 1024**3:.2f} GiB | free VRAM {free / 1024**3:.2f} GiB")

PORT = 8001
BASE_MODEL = MODEL_CONFIG["base_model"]
CONTROLNET_MODEL = MODEL_CONFIG["controlnet_model"]
ADAPTER_PATH = LORA_PATH
RANK = 8

def get_secret(name, required=True):
    value = None
    try:
        from kaggle_secrets import UserSecretsClient
        value = UserSecretsClient().get_secret(name)
    except Exception:
        value = None
    if not value and required:
        value = getpass(f"Enter {name} (input hidden; Kaggle Secrets recommended): ").strip()
    if required and not value:
        raise RuntimeError(f"Missing required secret: {name}")
    return value

NGROK_AUTHTOKEN = get_secret("NGROK_AUTHTOKEN")
GPU_BACKEND_TOKEN = get_secret("GPU_BACKEND_TOKEN")
HF_TOKEN = get_secret("HF_TOKEN", required=False)
print("Secrets loaded; values are hidden.")


In [ ]:
from kaggle_secrets import UserSecretsClient

client = UserSecretsClient()

# These two secrets are required to expose/authenticate the backend.
required_secrets = ["NGROK_AUTHTOKEN", "GPU_BACKEND_TOKEN"]
optional_secrets = ["HF_TOKEN"]

for name in required_secrets:
    try:
        value = client.get_secret(name)
        if not value or not value.strip():
            raise ValueError("empty secret")
        print(f"OK: required secret {name} is configured.")
    except Exception as exc:
        raise RuntimeError(
            f"Required Kaggle Secret {name} is missing/unavailable. "
            "Add it in Kaggle Secrets before continuing."
        ) from exc

for name in optional_secrets:
    try:
        value = client.get_secret(name)
        print(f"OK: optional secret {name} is configured." if value and value.strip()
              else f"Optional secret {name} is not configured; public model access will be attempted.")
    except Exception:
        print(f"Optional secret {name} is not configured; public model access will be attempted.")


## 4. Load SDXL Inpainting + Canny ControlNet + LoRA

In [ ]:
from diffusers import StableDiffusionXLControlNetInpaintPipeline, ControlNetModel
from peft import LoraConfig
from peft.utils import set_peft_model_state_dict
from safetensors.torch import load_file

# CPU offload reduces peak VRAM use on T4-class GPUs. Load only once per session.
controlnet = ControlNetModel.from_pretrained(
    CONTROLNET_MODEL, torch_dtype=torch.float16, token=HF_TOKEN or None
)
pipe = StableDiffusionXLControlNetInpaintPipeline.from_pretrained(
    BASE_MODEL, controlnet=controlnet, torch_dtype=torch.float16,
    use_safetensors=True, token=HF_TOKEN or None
)
pipe.enable_model_cpu_offload()
pipe.enable_attention_slicing()

# Match the adapter architecture used for the existing rank-8 checkpoint.
pipe.unet.add_adapter(LoraConfig(
    r=RANK, lora_alpha=RANK, init_lora_weights="gaussian",
    target_modules=["to_q", "to_k", "to_v", "to_out.0"],
))
state_dict = load_file(str(ADAPTER_PATH), device="cpu")
load_result = set_peft_model_state_dict(pipe.unet, state_dict)
print("Adapter state loaded.")
print("Unexpected adapter keys:", getattr(load_result, "unexpected_keys", []))
print("Pipeline loaded with CPU offload.")


## Automatic mask generation: Grounding DINO + SAM
The backend uses Grounding DINO to locate the requested existing object/placement region and Segment Anything (SAM) to turn the selected box into a pixel mask. These two models stay on CPU to reserve GPU memory for SDXL. Their weights are downloaded on first startup.
- If a user uploads a mask, that mask is used instead.
- If no mask is uploaded, automatic masking is enabled by default.
- The highest-confidence detection is selected. If multiple similar objects are present, the detector can still choose the wrong instance; use a manual mask when exact instance selection matters.
- For adding an object, target the existing placement region (for example, person's face when adding sunglasses).
- For removing an object, target the object to remove. The backend switches to a background-continuation prompt.
- If detection or segmentation fails, the API returns an explicit error. /v1/mask can be called to inspect the generated mask before editing.

### A. Load text-guided automatic masking (Grounding DINO + SAM)
This optional mask stage lets the API find the object named by the user and segment it. These models are kept on CPU to avoid competing with SDXL for GPU memory. The first run downloads their public model weights. Automatic masking can fail when the target is ambiguous, hidden, or not present; users can still upload their own mask.

Targeting note: target must be a concise phrase for an object that already exists in the image (for example, apple or person's face). For adding a new object such as glasses, the target is the existing placement region (the face), not the glasses themselves. The current implementation selects the highest-confidence detection; ambiguous multiple-object selection still needs a manual mask.

In [ ]:
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection, SamModel, SamProcessor

GROUNDING_DINO_ID = "IDEA-Research/grounding-dino-tiny"
SAM_ID = "facebook/sam-vit-base"

print("Loading text-guided object detector on CPU...")
grounding_processor = AutoProcessor.from_pretrained(GROUNDING_DINO_ID, token=HF_TOKEN or None)
grounding_model = AutoModelForZeroShotObjectDetection.from_pretrained(
    GROUNDING_DINO_ID, token=HF_TOKEN or None
).to("cpu").eval()

print("Loading Segment Anything on CPU...")
sam_processor = SamProcessor.from_pretrained(SAM_ID, token=HF_TOKEN or None)
sam_model = SamModel.from_pretrained(SAM_ID, token=HF_TOKEN or None).to("cpu").eval()
print("Automatic masking models loaded (CPU).")

## 5. Run a minimal inference smoke test
This test uses a synthetic image and mask to verify that the loaded pipeline can actually generate an output before the API is exposed. It is an integration check, not a model-quality evaluation.

In [ ]:
from PIL import ImageDraw

# Synthetic test image: neutral background with a simple object-like rectangle.
smoke_source = Image.new("RGB", (512, 512), (180, 180, 180))
draw = ImageDraw.Draw(smoke_source)
draw.rectangle((150, 130, 360, 390), fill=(90, 90, 90), outline=(25, 25, 25), width=8)
smoke_mask = Image.new("L", (512, 512), 0)
mask_draw = ImageDraw.Draw(smoke_mask)
mask_draw.rectangle((145, 125, 365, 395), fill=255)
smoke_edges = cv2.Canny(cv2.cvtColor(np.asarray(smoke_source), cv2.COLOR_RGB2GRAY), 100, 200)
smoke_control = Image.fromarray(smoke_edges).convert("RGB")

with torch.inference_mode():
    smoke_result = pipe(
        prompt="a realistic blue ceramic vase, studio photograph, detailed material",
        negative_prompt="blurry, distorted, low quality",
        image=smoke_source,
        mask_image=smoke_mask,
        control_image=smoke_control,
        num_inference_steps=2,
        guidance_scale=5.0,
        controlnet_conditioning_scale=0.75,
        strength=0.8,
        generator=torch.Generator(device="cpu").manual_seed(42),
    ).images[0]

SMOKE_OUTPUT = OUTPUT_ROOT / "inference_smoke_test.png"
smoke_result.save(SMOKE_OUTPUT)
assert smoke_result.size == (512, 512), f"Unexpected output size: {smoke_result.size}"
print("Inference smoke test PASSED")
print("Output:", SMOKE_OUTPUT)
print("Output dimensions:", smoke_result.size)
display(smoke_result)

## 6. Define authenticated FastAPI inference endpoints

In [ ]:
# HOTFIX: Robust Grounding DINO + SAM automatic masking
# Tries alternative target phrases and detections when SAM returns an empty mask.
# Does not reload models or restart the API server.

def _automatic_mask(src, target, prompt, operation="modify"):
    phrase = _normalise_target(target, prompt, operation)

    if not phrase:
        raise HTTPException(
            status_code=422,
            detail="Automatic masking needs a target description or a supplied mask."
        )

    low = phrase.lower()
    candidates = [phrase]

    # Try alternate descriptions even when the first phrase produces
    # a detection that SAM cannot segment.
    if any(x in low for x in ("face", "eye", "glasses", "sunglasses", "hat", "earring")):
        candidates += ["face", "head", "person"]

    if "head" in low:
        candidates += ["head", "person"]

    if "wall" in low:
        candidates += ["wall", "building wall"]

    candidates = list(dict.fromkeys(
        x.strip(" .") for x in candidates if x.strip()
    ))

    failures = []

    for candidate in candidates:
        try:
            detections = _dino_detect(src, candidate)
            boxes = detections.get("boxes")
            scores = detections.get("scores")

            if boxes is None or len(boxes) == 0:
                failures.append(f"{candidate}: no detection")
                continue

            # Try detections in descending confidence order rather than
            # trusting only the first box.
            if scores is not None and len(scores):
                order = torch.argsort(scores, descending=True).tolist()
            else:
                order = list(range(len(boxes)))

            for box_idx in order[:5]:
                box = boxes[box_idx].detach().cpu().tolist()

                # Reject invalid or degenerate bounding boxes.
                x1, y1, x2, y2 = box
                x1 = max(0.0, min(float(src.width), x1))
                x2 = max(0.0, min(float(src.width), x2))
                y1 = max(0.0, min(float(src.height), y1))
                y2 = max(0.0, min(float(src.height), y2))

                if x2 - x1 < 4 or y2 - y1 < 4:
                    failures.append(f"{candidate}: invalid bounding box")
                    continue

                box = [x1, y1, x2, y2]

                sam_inputs = sam_processor(
                    images=src,
                    input_boxes=[[box]],
                    return_tensors="pt",
                )

                with torch.inference_mode():
                    sam_outputs = sam_model(**sam_inputs)

                processed = sam_processor.post_process_masks(
                    sam_outputs.pred_masks.detach().cpu(),
                    sam_inputs["original_sizes"].cpu(),
                    sam_inputs["reshaped_input_sizes"].cpu(),
                )[0]

                candidate_masks = processed[0][0]
                iou_scores = sam_outputs.iou_scores.detach().cpu()[0, 0]

                mask_order = torch.argsort(
                    iou_scores, descending=True
                ).tolist()

                for mask_idx in mask_order:
                    mask_array = candidate_masks[mask_idx].numpy() > 0

                    if not mask_array.any():
                        continue

                    mask_u8 = mask_array.astype(np.uint8) * 255
                    mask_u8 = cv2.dilate(
                        mask_u8,
                        np.ones((5, 5), np.uint8),
                        iterations=1,
                    )

                    mask_img = Image.fromarray(
                        mask_u8, mode="L"
                    ).resize(
                        src.size,
                        Image.Resampling.NEAREST,
                    )

                    print(
                        f"Automatic mask succeeded: "
                        f"target={candidate!r}, box_index={box_idx}, "
                        f"mask_index={mask_idx}"
                    )

                    return mask_img, candidate

                failures.append(f"{candidate}: SAM returned empty masks")

        except HTTPException:
            raise
        except Exception as exc:
            failures.append(
                f"{candidate}: {type(exc).__name__}: {str(exc)[:120]}"
            )

    detail = "; ".join(failures[-12:])

    raise HTTPException(
        status_code=422,
        detail=(
            f"Could not create a usable mask for '{phrase}'. "
            f"Tried alternate detections. Details: {detail}. "
            "Try supplying an editable mask."
        ),
    )

print("Automatic-mask fallback hotfix installed.")

In [ ]:
import inspect
import re

from fastapi import FastAPI, File, Form, Header, HTTPException, UploadFile
from fastapi.responses import Response


# ============================================================
# 1. APPLICATION CONFIGURATION
# ============================================================

app = FastAPI(
    title="Part 1 GPU Backend",
    version="2.1.0-automatic-mask",
)

_inference_lock = threading.Lock()

GENERIC_TARGETS = {
    "",
    "object",
    "thing",
    "item",
    "the object",
    "object described in prompt",
}


# ============================================================
# 2. AUTHENTICATION AND IMAGE INPUT
# ============================================================

def _check_token(x_backend_token: str | None) -> None:
    """Require the configured backend authentication token."""
    if (
        not x_backend_token
        or not secrets.compare_digest(
            x_backend_token,
            GPU_BACKEND_TOKEN,
        )
    ):
        raise HTTPException(
            status_code=401,
            detail="Missing or invalid backend token",
        )


def _open_rgb(raw: bytes, name: str) -> Image.Image:
    """Decode an uploaded image as RGB."""
    try:
        return Image.open(io.BytesIO(raw)).convert("RGB")
    except Exception as exc:
        raise HTTPException(
            status_code=400,
            detail=f"Invalid {name} image",
        ) from exc


def _open_mask(raw: bytes) -> Image.Image:
    """Decode an uploaded mask as grayscale."""
    try:
        return Image.open(io.BytesIO(raw)).convert("L")
    except Exception as exc:
        raise HTTPException(
            status_code=400,
            detail="Invalid mask image",
        ) from exc


# ============================================================
# 3. TARGET PHRASE NORMALIZATION
# ============================================================

def _normalise_target(
    target: str,
    prompt: str,
    operation: str,
) -> str:
    """
    Resolve a concise visual target for Grounding DINO.

    For add operations, prefer the existing placement region
    over the object that the model is being asked to add.
    """
    raw = (target or "").strip().strip(" .")

    if raw.lower() not in GENERIC_TARGETS:
        return raw[:160]

    text = (prompt or "").strip().strip(" .")

    patterns = [
        # Example: "Add sunglasses to the person's face"
        r"\badd\s+.+?\s+to\s+(?:the\s+)?(.+)$",

        # Example: "Add sunglasses over the person's eyes"
        r"\badd\s+.+?\s+(?:over|onto|on|around|across|above|under|beneath)\s+(?:the\s+)?(.+)$",

        # Example: "Put a hat on the person's head"
        r"\bput\s+.+?\s+on\s+(?:the\s+)?(.+)$",

        # Example: "Place a lamp on the table"
        r"\bplace\s+.+?\s+on\s+(?:the\s+)?(.+)$",

        # Example: "Remove the red car"
        r"\b(?:remove|delete|erase)\s+(?:the|a|an)?\s*(.+)$",

        # Example: "Replace the chair with a sofa"
        r"\b(?:replace|change|turn|paint|color|colour)\s+"
        r"(?:the|a|an)?\s*(.+?)\s+(?:with|into|to)\s+.+$",

        # Example: "Make the car red"
        r"\bmake\s+(?:the|a|an)?\s*(.+?)\s+"
        r"(?:red|blue|green|yellow|black|white|brown|pink|purple|"
        r"orange|silver|gold|wooden|metallic)\b",
    ]

    for pattern in patterns:
        match = re.search(
            pattern,
            text,
            flags=re.IGNORECASE,
        )

        if match:
            candidate = match.group(1).strip(" .,!?:;")

            if candidate:
                return candidate[:160]

    return text[:160]


# ============================================================
# 4. GROUNDING DINO POST-PROCESSING COMPATIBILITY
# ============================================================

def _post_process_grounding_detections(
    outputs,
    input_ids,
    target_sizes,
    box_threshold,
    text_threshold,
):
    """
    Support Transformers versions using either:
        box_threshold=
    or:
        threshold=

    Also handle wrappers that expose incomplete signatures.
    """
    fn = grounding_processor.post_process_grounded_object_detection

    try:
        params = inspect.signature(fn).parameters
    except (TypeError, ValueError):
        params = {}

    accepts_kwargs = any(
        parameter.kind == inspect.Parameter.VAR_KEYWORD
        for parameter in params.values()
    )

    kwargs = {}

    if (
        not params
        or "target_sizes" in params
        or accepts_kwargs
    ):
        kwargs["target_sizes"] = target_sizes

    if "box_threshold" in params:
        kwargs["box_threshold"] = box_threshold
    elif "threshold" in params:
        kwargs["threshold"] = box_threshold
    else:
        kwargs["threshold"] = box_threshold

    if (
        not params
        or "text_threshold" in params
        or accepts_kwargs
    ):
        kwargs["text_threshold"] = text_threshold

    accepts_input_ids = (
        not params
        or "input_ids" in params
        or accepts_kwargs
    )

    try:
        if accepts_input_ids:
            return fn(
                outputs,
                input_ids=input_ids,
                **kwargs,
            )

        return fn(outputs, **kwargs)

    except TypeError as first_error:
        # Fallback for wrappers whose signatures are misleading.
        variants = [
            {
                "box_threshold": box_threshold,
                "text_threshold": text_threshold,
                "target_sizes": target_sizes,
            },
            {
                "threshold": box_threshold,
                "text_threshold": text_threshold,
                "target_sizes": target_sizes,
            },
        ]

        for variant in variants:
            for include_input_ids in (True, False):
                try:
                    if include_input_ids:
                        return fn(
                            outputs,
                            input_ids=input_ids,
                            **variant,
                        )

                    return fn(outputs, **variant)

                except TypeError:
                    continue

        raise RuntimeError(
            "Unsupported Grounding DINO post-processing API: "
            f"{first_error}"
        ) from first_error


# ============================================================
# 5. GROUNDING DINO DETECTION
# ============================================================

def _dino_detect(
    src: Image.Image,
    phrase: str,
):
    """Detect the requested visual target."""
    text = phrase.lower().strip()

    if not text.endswith("."):
        text += "."

    det_inputs = grounding_processor(
        images=src,
        text=text,
        return_tensors="pt",
    )

    # Keep input tensors on the same device as Grounding DINO.
    model_device = next(grounding_model.parameters()).device

    det_inputs = {
        key: (
            value.to(model_device)
            if torch.is_tensor(value)
            else value
        )
        for key, value in det_inputs.items()
    }

    with torch.inference_mode():
        det_outputs = grounding_model(**det_inputs)

        detections = _post_process_grounding_detections(
            det_outputs,
            det_inputs.get("input_ids"),
            target_sizes=[(src.height, src.width)],
            box_threshold=0.22,
            text_threshold=0.15,
        )[0]

    return detections


# ============================================================
# 6. AUTOMATIC MASK GENERATION: GROUNDING DINO + SAM
# ============================================================

def _automatic_mask(
    src: Image.Image,
    target: str,
    prompt: str,
    operation: str = "modify",
) -> tuple[Image.Image, str]:
    """
    Detect a target using Grounding DINO, segment its bounding
    box with SAM, and return a white-on-black binary mask.

    White = editable.
    Black = preserve original pixels.
    """
    phrase = _normalise_target(
        target,
        prompt,
        operation,
    )

    if not phrase:
        raise HTTPException(
            status_code=422,
            detail=(
                "Automatic masking needs a target description. "
                "Name the object or placement region, or upload a mask."
            ),
        )

    # Always try the resolved target before fallback phrases.
    candidates = [phrase]
    low = phrase.lower()

    if any(
        term in low
        for term in (
            "face",
            "eye",
            "glasses",
            "sunglasses",
            "hat",
            "earring",
        )
    ):
        candidates.extend(["face", "head", "person"])

    if "head" in low:
        candidates.extend(["head", "face", "person"])

    if "wall" in low:
        candidates.extend(["wall", "building wall"])

    words = phrase.split()

    if len(words) > 5:
        candidates.append(" ".join(words[:5]))

    candidates = list(
        dict.fromkeys(
            candidate.strip(" .")
            for candidate in candidates
            if candidate.strip(" .")
        )
    )

    detections = None
    selected_phrase = phrase

    try:
        # ----------------------------------------------------
        # 6A. Locate the target
        # ----------------------------------------------------

        for candidate in candidates:
            result = _dino_detect(src, candidate)

            boxes = result.get("boxes")
            scores = result.get("scores")

            if boxes is not None and len(boxes) > 0:
                detections = result
                selected_phrase = candidate
                break

        if detections is None:
            raise HTTPException(
                status_code=422,
                detail=(
                    f"Could not locate '{phrase}' in the image. "
                    "Try a visible object or provide a mask."
                ),
            )

        boxes = detections["boxes"]
        scores = detections.get("scores")

        if scores is not None and len(scores) > 0:
            best_detection_idx = int(
                torch.argmax(scores).item()
            )
        else:
            best_detection_idx = 0

        box = (
            boxes[best_detection_idx]
            .detach()
            .cpu()
            .tolist()
        )

        # ----------------------------------------------------
        # 6B. Run SAM on the selected bounding box
        # ----------------------------------------------------

        sam_inputs = sam_processor(
            images=src,
            input_boxes=[[box]],
            return_tensors="pt",
        )

        sam_device = next(sam_model.parameters()).device

        sam_inputs = {
            key: (
                value.to(sam_device)
                if torch.is_tensor(value)
                else value
            )
            for key, value in sam_inputs.items()
        }

        with torch.inference_mode():
            sam_outputs = sam_model(**sam_inputs)

        # Move predicted masks to CPU before post-processing.
        raw_masks = sam_outputs.pred_masks.detach().cpu()

        processed_masks = sam_processor.post_process_masks(
            raw_masks,
            sam_inputs["original_sizes"].detach().cpu(),
            sam_inputs["reshaped_input_sizes"].detach().cpu(),
        )

        if not processed_masks:
            raise HTTPException(
                status_code=422,
                detail="SAM did not return any processed masks.",
            )

        # ----------------------------------------------------
        # 6C. CORRECT SAM CANDIDATE SELECTION
        # ----------------------------------------------------
        #
        # Observed output for this backend:
        #
        # processed_masks[0].shape = [1, 3, H, W]
        # iou_scores.shape          = [1, 1, 3]
        #
        # First index selects the input image.
        # Second index selects its bounding box.
        # Third index selects one of SAM's candidate masks.
        #
        # The old code indexed masks[0][0] and then indexed it
        # again as if the candidate dimension still existed.
        # That was the tensor-indexing bug.
        # ----------------------------------------------------

        image_masks = processed_masks[0]

        # Remove the batch dimension, retaining all candidates.
        if image_masks.ndim == 4:
            candidate_masks = image_masks[0]
        elif image_masks.ndim == 3:
            candidate_masks = image_masks
        else:
            raise HTTPException(
                status_code=500,
                detail=(
                    "Unexpected SAM mask dimensions: "
                    f"{tuple(image_masks.shape)}"
                ),
            )

        if candidate_masks.shape[0] == 0:
            raise HTTPException(
                status_code=422,
                detail="SAM returned zero candidate masks.",
            )

        iou_values = (
            sam_outputs.iou_scores
            .detach()
            .cpu()
        )

        # Expected IoU dimensions: [batch, boxes, candidates].
        if iou_values.ndim == 3:
            candidate_scores = iou_values[0, 0]
        elif iou_values.ndim == 2:
            candidate_scores = iou_values[0]
        elif iou_values.ndim == 1:
            candidate_scores = iou_values
        else:
            raise HTTPException(
                status_code=500,
                detail=(
                    "Unexpected SAM IoU dimensions: "
                    f"{tuple(iou_values.shape)}"
                ),
            )

        if len(candidate_scores) != candidate_masks.shape[0]:
            raise HTTPException(
                status_code=500,
                detail=(
                    "SAM candidate count does not match IoU scores: "
                    f"{candidate_masks.shape[0]} masks versus "
                    f"{len(candidate_scores)} scores."
                ),
            )

        # Choose the candidate with the highest predicted IoU.
        best_mask_idx = int(
            torch.argmax(candidate_scores).item()
        )

        selected_mask = candidate_masks[best_mask_idx]

        # The observed Transformers version returns boolean
        # post-processed masks. Retain compatibility with
        # versions that return numeric mask logits.
        if selected_mask.dtype == torch.bool:
            mask_array = selected_mask.numpy()
        else:
            mask_array = (
                selected_mask.numpy() > 0
            )

        if mask_array.ndim != 2:
            raise HTTPException(
                status_code=500,
                detail=(
                    "Selected SAM mask is not two-dimensional: "
                    f"{mask_array.shape}"
                ),
            )

        if not mask_array.any():
            raise HTTPException(
                status_code=422,
                detail=(
                    "SAM selected an empty mask despite "
                    "producing candidate masks."
                ),
            )

        selected_iou = float(
            candidate_scores[best_mask_idx].item()
        )

        print(
            "\n========== AUTOMATIC MASK RESULT ==========",
            flush=True,
        )
        print(f"Source size: {src.size}", flush=True)
        print(f"Requested target: {phrase}", flush=True)
        print(f"Selected target: {selected_phrase}", flush=True)
        print(f"Selected box: {box}", flush=True)
        print(
            f"SAM candidate: {best_mask_idx}",
            flush=True,
        )
        print(
            f"SAM predicted IoU: {selected_iou:.4f}",
            flush=True,
        )
        print(
            f"Mask dimensions: {mask_array.shape}",
            flush=True,
        )
        print(
            f"Editable pixels before dilation: "
            f"{int(mask_array.sum())}",
            flush=True,
        )
        print(
            "============================================\n",
            flush=True,
        )

        # ----------------------------------------------------
        # 6D. Build the final white-on-black mask
        # ----------------------------------------------------

        mask_u8 = mask_array.astype(np.uint8) * 255

        # Slightly expand the editable boundary to reduce
        # untouched outlines around the selected object.
        mask_u8 = cv2.dilate(
            mask_u8,
            np.ones((5, 5), np.uint8),
            iterations=1,
        )

        mask_img = Image.fromarray(
            mask_u8,
            mode="L",
        )

        # Ensure the final mask matches the original image.
        if mask_img.size != src.size:
            mask_img = mask_img.resize(
                src.size,
                Image.Resampling.NEAREST,
            )

        return mask_img, selected_phrase

    except HTTPException:
        raise

    except Exception as exc:
        raise HTTPException(
            status_code=500,
            detail=(
                "Automatic masking failed "
                f"({type(exc).__name__}): {str(exc)[:240]}"
            ),
        ) from exc


# ============================================================
# 7. INFERENCE IMAGE SIZE
# ============================================================

def _inference_size(
    width: int,
    height: int,
    max_side: int = 512,
) -> tuple[int, int]:
    """
    Downscale without changing aspect ratio.

    Round each dimension to a multiple of eight for
    diffusion-model compatibility.
    """
    scale = min(
        max_side / max(width, height),
        1.0,
    )

    def rounded(value):
        return max(
            64,
            int(round((value * scale) / 8.0)) * 8,
        )

    return rounded(width), rounded(height)


# ============================================================
# 8. PNG RESPONSE HELPER
# ============================================================

def _png_response(
    image: Image.Image,
    headers: dict | None = None,
) -> Response:
    """Encode an image as a PNG HTTP response."""
    buf = io.BytesIO()

    image.save(
        buf,
        format="PNG",
    )

    return Response(
        content=buf.getvalue(),
        media_type="image/png",
        headers=headers or {},
    )


# ============================================================
# 9. HEALTH ENDPOINT
# ============================================================

@app.get("/health")
def health(
    x_backend_token: str | None = Header(default=None),
):
    """Authenticated backend health and model-status check."""
    _check_token(x_backend_token)

    return {
        "status": "ok",
        "service": "sag-part1-gpu-backend",
        "gpu": torch.cuda.get_device_name(0),
        "adapter": ADAPTER_PATH.name,
        "model_loaded": pipe is not None,
        "automatic_masking": (
            grounding_model is not None
            and sam_model is not None
        ),
        "endpoints": [
            "/health",
            "/v1/mask",
            "/v1/edit",
        ],
    }


# ============================================================
# 10. MASK PREVIEW ENDPOINT
# ============================================================

@app.post("/v1/mask")
async def create_mask(
    image: UploadFile = File(...),
    target: str = Form("object"),
    prompt: str = Form(""),
    operation: str = Form("modify"),
    x_backend_token: str | None = Header(default=None),
):
    """
    Return the automatic mask without running diffusion.

    White pixels are editable.
    Black pixels are preserved.
    """
    _check_token(x_backend_token)

    src = _open_rgb(
        await image.read(),
        "source",
    )

    if src.width < 64 or src.height < 64:
        raise HTTPException(
            status_code=400,
            detail="Source image must be at least 64x64 pixels",
        )

    mask_img, selected_target = _automatic_mask(
        src,
        target,
        prompt,
        operation,
    )

    return _png_response(
        mask_img,
        {
            "X-Mask-Mode": "automatic",
            "X-Mask-Target": selected_target,
        },
    )


# ============================================================
# 11. FULL IMAGE EDITING ENDPOINT
# ============================================================

@app.post("/v1/edit")
async def edit_image(
    image: UploadFile = File(...),
    mask: UploadFile | None = File(None),
    prompt: str = Form(...),
    target: str = Form("object"),
    operation: str = Form("modify"),
    auto_mask: bool = Form(True),
    x_backend_token: str | None = Header(default=None),
    negative_prompt: str = Form(
        "cartoon, illustration, painting, CGI, 3d render, "
        "distorted geometry, blurry, low quality, artifacts, "
        "text, watermark"
    ),
    num_inference_steps: int = Form(25),
    guidance_scale: float = Form(6.0),
    controlnet_conditioning_scale: float = Form(0.75),
    strength: float = Form(0.8),
    seed: int = Form(42),
):
    """
    Image-editing pipeline:

        Image + instruction
            -> Automatic mask or uploaded mask
            -> SDXL inpainting + ControlNet + LoRA
            -> Restore original pixels outside mask
            -> PNG response
    """
    _check_token(x_backend_token)

    # --------------------------------------------------------
    # 11A. Validate request parameters
    # --------------------------------------------------------

    if not prompt or not prompt.strip():
        raise HTTPException(
            status_code=400,
            detail="prompt must not be empty",
        )

    if not 1 <= num_inference_steps <= 40:
        raise HTTPException(
            status_code=400,
            detail="num_inference_steps must be between 1 and 40",
        )

    if not 1.0 <= guidance_scale <= 20.0:
        raise HTTPException(
            status_code=400,
            detail="guidance_scale must be between 1 and 20",
        )

    if not 0.0 <= strength <= 1.0:
        raise HTTPException(
            status_code=400,
            detail="strength must be between 0 and 1",
        )

    if not 0.0 <= controlnet_conditioning_scale <= 2.0:
        raise HTTPException(
            status_code=400,
            detail=(
                "controlnet_conditioning_scale must be "
                "between 0 and 2"
            ),
        )

    if operation not in {
        "modify",
        "replace",
        "add",
        "remove",
    }:
        raise HTTPException(
            status_code=400,
            detail=(
                "operation must be modify, replace, add, or remove"
            ),
        )

    # --------------------------------------------------------
    # 11B. Load and validate source image
    # --------------------------------------------------------

    src = _open_rgb(
        await image.read(),
        "source",
    )

    if src.width < 64 or src.height < 64:
        raise HTTPException(
            status_code=400,
            detail="Source image must be at least 64x64 pixels",
        )

    # --------------------------------------------------------
    # 11C. Obtain the editing mask
    # --------------------------------------------------------

    if mask is not None:
        mask_bytes = await mask.read()

        if not mask_bytes:
            raise HTTPException(
                status_code=400,
                detail="Uploaded mask is empty",
            )

        mask_img = _open_mask(mask_bytes)

        if mask_img.size != src.size:
            mask_img = mask_img.resize(
                src.size,
                Image.Resampling.NEAREST,
            )

        selected_target = (target or "manual mask")
        mask_mode = "provided"

    elif auto_mask:
        mask_img, selected_target = _automatic_mask(
            src,
            target,
            prompt,
            operation,
        )

        mask_mode = "automatic"

    else:
        raise HTTPException(
            status_code=422,
            detail=(
                "No mask supplied. Upload a mask or enable "
                "automatic masking."
            ),
        )

    mask_arr_native = np.asarray(mask_img) > 127

    if not mask_arr_native.any():
        raise HTTPException(
            status_code=422,
            detail="Mask contains no editable (white) pixels",
        )

    if mask_arr_native.all():
        raise HTTPException(
            status_code=422,
            detail=(
                "Mask covers the entire image; select a "
                "smaller target region."
            ),
        )

    # --------------------------------------------------------
    # 11D. Prepare diffusion inputs
    # --------------------------------------------------------

    work_w, work_h = _inference_size(
        src.width,
        src.height,
    )

    src_work = src.resize(
        (work_w, work_h),
        Image.Resampling.LANCZOS,
    )

    mask_work = mask_img.resize(
        (work_w, work_h),
        Image.Resampling.NEAREST,
    )

    src_arr_work = np.asarray(src_work)

    gray_work = cv2.cvtColor(
        src_arr_work,
        cv2.COLOR_RGB2GRAY,
    )

    edge = cv2.Canny(
        gray_work,
        100,
        200,
    )

    control_img = Image.fromarray(
        edge,
    ).convert("RGB")

    # --------------------------------------------------------
    # 11E. Construct operation-aware prompts
    # --------------------------------------------------------

    if operation == "remove":
        prompt_full = (
            "photorealistic continuation of the surrounding "
            f"background where {selected_target} was removed, "
            "seamless texture, consistent perspective and lighting, "
            "empty region, no object remaining"
        )

        negative_full = (
            f"{negative_prompt}, "
            f"{selected_target}, duplicate object"
        )

    else:
        prompt_full = (
            prompt.strip()
            + ", realistic photograph, natural light, "
              "real material, detailed texture"
        )

        negative_full = negative_prompt

    # --------------------------------------------------------
    # 11F. Run SDXL inpainting pipeline
    # --------------------------------------------------------

    try:
        with _inference_lock, torch.inference_mode():
            result = pipe(
                prompt=prompt_full,
                negative_prompt=negative_full,
                image=src_work,
                mask_image=mask_work,
                control_image=control_img,
                num_inference_steps=num_inference_steps,
                guidance_scale=guidance_scale,
                controlnet_conditioning_scale=(
                    controlnet_conditioning_scale
                ),
                strength=strength,
                generator=torch.Generator(
                    device="cpu"
                ).manual_seed(int(seed)),
            ).images[0].convert("RGB")

        # ----------------------------------------------------
        # 11G. Restore original pixels outside the mask
        # ----------------------------------------------------
        #
        # This ensures the final image retains the exact
        # original RGB values wherever the native-resolution
        # mask is black.
        # ----------------------------------------------------

        generated_native = result.resize(
            src.size,
            Image.Resampling.LANCZOS,
        )

        original = np.asarray(src)
        generated = np.asarray(generated_native)

        output = original.copy()

        output[mask_arr_native] = generated[
            mask_arr_native
        ]

        response_image = Image.fromarray(output)

        # ----------------------------------------------------
        # 11H. Return edited PNG and metadata
        # ----------------------------------------------------

        return _png_response(
            response_image,
            {
                "X-Seed": str(seed),
                "X-Mask-Mode": mask_mode,
                "X-Mask-Target": selected_target[:120],
                "X-Operation": operation,
            },
        )

    except HTTPException:
        raise

    except torch.cuda.OutOfMemoryError as exc:
        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        raise HTTPException(
            status_code=503,
            detail=(
                "GPU ran out of memory during inference. "
                "Retry with a smaller source image or fewer "
                "inference steps."
            ),
        ) from exc

    except Exception as exc:
        raise HTTPException(
            status_code=500,
            detail=(
                f"Inference failed ({type(exc).__name__}): "
                f"{str(exc)[:240]}"
            ),
        ) from exc


# ============================================================
# 12. ROUTE REGISTRATION CONFIRMATION
# ============================================================

print("=" * 72)
print("PART 1 GPU BACKEND — ROUTES DEFINED")
print("=" * 72)
print("Version: 2.1.0-automatic-mask")
print("Authenticated endpoints:")
print("  GET  /health")
print("  POST /v1/mask")
print("  POST /v1/edit")
print()
print("Automatic masking:")
print("  Grounding DINO -> bounding box -> SAM candidates")
print("  -> highest-IoU candidate -> binary mask")
print()
print("Image editing:")
print("  Source image + mask + instruction")
print("  -> SDXL inpainting + ControlNet + LoRA")
print("  -> restore original pixels outside mask")
print("  -> PNG response")
print("=" * 72)

## 7. Start FastAPI and expose an HTTPS tunnel

In [ ]:
import os
import time
import threading
import urllib.request
import urllib.error
import uvicorn
from pyngrok import ngrok, conf
from kaggle_secrets import UserSecretsClient

# Fetch the secret securely at startup so a corrected Kaggle Secret is used.
NGROK_AUTHTOKEN = UserSecretsClient().get_secret("NGROK_AUTHTOKEN").strip()
if not NGROK_AUTHTOKEN or any(ch.isspace() for ch in NGROK_AUTHTOKEN):
    raise RuntimeError(
        "NGROK_AUTHTOKEN is empty or contains whitespace. In Kaggle Secrets, "
        "paste only the actual ngrok authtoken from your ngrok dashboard—not a URL or error message."
    )

# Also expose GPU_BACKEND_TOKEN to the environment variables if available
try:
    os.environ["GPU_BACKEND_TOKEN"] = UserSecretsClient().get_secret("GPU_BACKEND_TOKEN").strip()
except Exception:
    pass

# Clean up any failed/stale ngrok process, then apply the corrected token.
try:
    ngrok.kill()
except Exception:
    pass
ngrok.set_auth_token(NGROK_AUTHTOKEN)
conf.get_default().auth_token = NGROK_AUTHTOKEN

# Uvicorn runs on a daemon thread, so this cell can finish and later cells can run.
# It remains alive only while this Kaggle kernel/session is alive.
if "uvicorn_thread" in globals() and uvicorn_thread.is_alive():
    print("Uvicorn thread already running; reusing it.")
else:
    uvicorn_config = uvicorn.Config(
        app, host="0.0.0.0", port=PORT, log_level="warning", access_log=False
    )
    uvicorn_server = uvicorn.Server(uvicorn_config)
    uvicorn_thread = threading.Thread(
        target=uvicorn_server.run, daemon=True, name="gpu-backend-uvicorn"
    )
    uvicorn_thread.start()

ready = False
for _ in range(60):
    try:
        req = urllib.request.Request(f"http://127.0.0.1:{PORT}/health")
        urllib.request.urlopen(req, timeout=1)
    except urllib.error.HTTPError as exc:
        # A 401 is expected here because this readiness probe intentionally omits the token.
        if exc.code == 401:
            ready = True
            break
        time.sleep(1)
    except Exception:
        time.sleep(1)
    else:
        ready = True
        break
if not ready:
    raise RuntimeError("Uvicorn did not become ready. Check earlier cell output.")

# FIXED: Removed the deprecated parameters causing 'field schemes not found in type config.Endpoint' 
# Modern pyngrok natively handles standard HTTP traffic configurations directly from the port number.
tunnel = ngrok.connect(PORT)
PUBLIC_URL = tunnel.public_url

if not PUBLIC_URL.startswith("https://"):
    raise RuntimeError("Expected an HTTPS public URL from ngrok.")
    
print("GPU backend started.")
print("Public HTTPS base URL:", PUBLIC_URL)
print("Health endpoint:", PUBLIC_URL + "/health")
print("Edit endpoint:", PUBLIC_URL + "/v1/edit")
print("Keep this Kaggle session running. Never share the backend token.")


## 8. Verify authenticated health endpoint

In [ ]:
import requests
health_response = requests.get(
    PUBLIC_URL + "/health",
    headers={"X-Backend-Token": GPU_BACKEND_TOKEN},
    timeout=30,
)
print("HTTP status:", health_response.status_code)
health_response.raise_for_status()
print(health_response.json())

In [ ]:
print("Hello Check")

## 9. Connect the local app

On your local machine, set the endpoint URL and the **same** backend token in your environment (do not commit it to source control):

```bash
export GPU_BACKEND_URL="https://YOUR_NGROK_HOST/v1/edit"
export GPU_BACKEND_TOKEN="YOUR_SAME_RANDOM_SECRET"
```

The local image/manipulator.py must send X-Backend-Token as an HTTP header and send multipart image, mask, and prompt fields. Restart your local FastAPI service and UI after updating its client.
### Operational limitations
- The URL can change when the Kaggle session or tunnel restarts.
- The Kaggle session must stay active. This is a release candidate, not persistent hosting.
- The endpoint performs masked image edits only; it does not automatically detect/segment objects or independently implement every add/remove/replace workflow.
- The adapter has only 20 training steps and is not validated for production-quality edits. A successful health check confirms service availability, not output quality.
- Inference uses a 512×512 square canvas and may distort aspect ratio during generation. Outside-mask pixels are restored from the source imag

## Local orchestrator integration contract

After this notebook starts successfully, copy the printed HTTPS tunnel URL into the local `.env` as `GPU_BACKEND_URL=<PUBLIC_URL>/v1/edit`, and set `GPU_BACKEND_TOKEN` to the exact same secret used by this notebook. The local LangGraph framework plans and evaluates tasks; this Kaggle service only performs image inference.

The local worker expects `POST /v1/edit` to accept multipart fields for the source image and edit prompt, with optional mask and task parameters, and to return an image response. Verify the local app's health/capability endpoints separately; a successful Kaggle `/health` check confirms only backend availability, not edit quality or full agent-run success.
